# FB-CLIP zero-shot natural-corruption benchmark

Run with a Kaggle GPU and Internet enabled. The notebook uses the official FB-CLIP implementation and released Google Drive checkpoint, while the shared harness evaluates the clean baseline and natural-corruption conditions. It enforces cross-dataset zero-shot evaluation: VisA-trained weights evaluate MVTec AD, and MVTec-trained weights evaluate VisA.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("====== STEP 1: CLONING BENCHMARK AND OFFICIAL FB-CLIP ======")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
FBCLIP_ROOT = Path("/kaggle/working/FB-CLIP")

def clone_or_update(url, destination):
    if not destination.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", url, str(destination)],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(destination), "pull", "--ff-only"],
            check=True,
        )

clone_or_update(
    f"https://github.com/{BENCHMARK_REPOSITORY}.git", BENCHMARK_ROOT
)
clone_or_update("https://github.com/Xi-Mu-Yu/FB-CLIP.git", FBCLIP_ROOT)

print("\n====== STEP 2: INSTALLING FB-CLIP DEPENDENCIES ======")
# Keep Kaggle's CUDA-enabled torch/torchvision instead of replacing them
# with the upstream CUDA 11.8 pins.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "gdown>=5.2", "setuptools<81", "ftfy>=6.1", "regex>=2023.0",
        "tqdm>=4.64", "scipy>=1.9",
        "opencv-python-headless>=4.8", "scikit-image>=0.20",
        "scikit-learn>=1.2", "matplotlib>=3.7",
        "seaborn>=0.12", "tabulate>=0.9", "Wand>=0.6",
    ],
    check=True,
)
required_files = [
    FBCLIP_ROOT / "FBCLIP_lib" / "FBCLIP.py",
    FBCLIP_ROOT / "prompt_ensemble.py",
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Incomplete FB-CLIP clone; missing: {missing_files}")
os.environ["FBCLIP_ROOT"] = str(FBCLIP_ROOT)
print(f"Benchmark repository: {BENCHMARK_ROOT}")
print(f"Official FB-CLIP:      {FBCLIP_ROOT}")
print("Environment ready.")

In [ ]:
import gc
import hashlib
import os
import sys
import urllib.request
from pathlib import Path

import gdown
import torch

BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
HARNESS_ROOT = BENCHMARK_ROOT / "zero_shot"
FBCLIP_ROOT = Path("/kaggle/working/FB-CLIP")
for import_path in (BENCHMARK_ROOT, HARNESS_ROOT, FBCLIP_ROOT):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ["FBCLIP_ROOT"] = str(FBCLIP_ROOT)

from shared import corruption_plan_path
from harness.config import CLEAN_CONDITION
from harness.runner import run_evaluation

# Choose exactly one evaluation target.
# DATASET_NAME = "mvtec"
DATASET_NAME = "visa"
MODEL_NAME = "FB-CLIP"
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa"}:
    raise ValueError("DATASET_NAME must be either 'mvtec' or 'visa'.")
IS_MVTEC = DATASET_NAME == "mvtec"
SELECTED_DATASET = "MVTec AD" if IS_MVTEC else "VisA"
# Paper-faithful zero-shot protocol: always evaluate the other domain.
WEIGHT_DATASET = "visa" if IS_MVTEC else "mvtec"
if WEIGHT_DATASET == DATASET_NAME:
    raise RuntimeError("FB-CLIP zero-shot checkpoint leakage detected.")

USE_CATEGORIZED_CORRUPTIONS = True
CATEGORIZED_CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "motion_blur", "zoom_blur",
    "brightness", "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = [
    "noise", "blur", "photometric", "geometric"
]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS
    else UNCATEGORIZED_CORRUPTION_TYPES
)
INCLUDE_CLEAN_BASELINE = True
ZERO_CORRUPTION_CONDITION = CLEAN_CONDITION
SEVERITY_LEVELS = [1, 2, 3, 4]
BATCH_SIZE = 1  # Safe default for a Kaggle T4; increase only after a smoke run.
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"

MVTEC_PATH = (
    "/kaggle/input/datasets/alirezasalehy/mvtec-ad/"
    "mvtec_anomaly_detection"
)
VISA_PATH = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Enable a Kaggle GPU accelerator for FB-CLIP.")

CORRUPTION_PLAN = corruption_plan_path(DATASET_NAME)
if USE_CATEGORIZED_CORRUPTIONS and not CORRUPTION_PLAN.exists():
    raise FileNotFoundError(
        f"Categorized corruption plan not found: {CORRUPTION_PLAN}"
    )

# Exact file IDs from the public checkpoint folder linked by FB-CLIP.
CHECKPOINT_SPECS = {
    "mvtec": (
        "mvtec_epoch_1_model.pth",
        "1Qw0w-5WeYcVbOlQvrJnjcAgjP9SLhMTC",
    ),
    "visa": (
        "visa_epoch_2_model.pth",
        "1hzKUafDEpF1KUk6psKnA3anrAGR2nGj4",
    ),
}
CHECKPOINT_DIR = Path("/kaggle/working/fbclip_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def find_attached_file(filename):
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    return next(input_root.rglob(filename), None)

def resolve_released_checkpoint(weight_dataset):
    filename, file_id = CHECKPOINT_SPECS[weight_dataset]
    attached = find_attached_file(filename)
    if attached is not None:
        return attached
    destination = CHECKPOINT_DIR / filename
    if not destination.is_file():
        print(f"Downloading official FB-CLIP checkpoint: {filename}")
        result = gdown.download(
            id=file_id, output=str(destination), quiet=False
        )
        if result is None:
            raise RuntimeError(f"gdown could not download {filename}.")
    if destination.stat().st_size < 10_000_000:
        raise RuntimeError(
            f"Downloaded checkpoint is unexpectedly small: {destination}"
        )
    return destination

FBCLIP_CHECKPOINT = resolve_released_checkpoint(WEIGHT_DATASET)
if not FBCLIP_CHECKPOINT.name.startswith(f"{WEIGHT_DATASET}_epoch_"):
    raise RuntimeError(
        "FB-CLIP checkpoint filename does not match WEIGHT_DATASET: "
        f"{FBCLIP_CHECKPOINT}"
    )

OPENAI_CLIP_URL = (
    "https://openaipublic.azureedge.net/clip/models/"
    "3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02/"
    "ViT-L-14-336px.pt"
)
OPENAI_CLIP_SHA256 = (
    "3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02"
)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file_obj:
        for chunk in iter(lambda: file_obj.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def resolve_clip_weight():
    attached = find_attached_file("ViT-L-14-336px.pt")
    if attached is not None:
        return attached
    destination = FBCLIP_ROOT / "clip" / "ViT-L-14-336px.pt"
    if not destination.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_suffix(".pt.download")
        print("Downloading OpenAI ViT-L/14@336px backbone...")
        urllib.request.urlretrieve(OPENAI_CLIP_URL, temporary)
        if sha256_file(temporary) != OPENAI_CLIP_SHA256:
            temporary.unlink(missing_ok=True)
            raise RuntimeError("OpenAI CLIP download failed SHA-256 validation.")
        temporary.replace(destination)
    elif sha256_file(destination) != OPENAI_CLIP_SHA256:
        raise RuntimeError(f"Existing CLIP weight failed SHA-256: {destination}")
    return destination

FBCLIP_CLIP_WEIGHT = resolve_clip_weight()
model_kwargs = {
    MODEL_NAME: {
        "fbclip_root": str(FBCLIP_ROOT),
        "checkpoint_path": str(FBCLIP_CHECKPOINT),
        "dataset_name": DATASET_NAME,
        "weight_dataset": WEIGHT_DATASET,
        "clip_weight_path": str(FBCLIP_CLIP_WEIGHT),
        "clip_download_dir": str(FBCLIP_ROOT / "clip"),
        "clip_model_name": "ViT-L/14@336px",
        "image_size": 518,
        "depth": 9,
        "n_ctx": 12,
        "t_n_ctx": 4,
        "feature_map_layer": [5, 11, 17, 24],
        "features_list": [5, 11, 17, 24],
        "feature_layers": [1, 6, 12, 18, 24],
        "sigma": 4,
        "use_gaussian_filter": True,
    }
}

print("LAUNCHING FB-CLIP ROBUSTNESS BENCHMARK")
print(f"Evaluation target: {SELECTED_DATASET}")
print(f"Weights trained on: {WEIGHT_DATASET} (cross-dataset zero-shot)")
print(f"Checkpoint:         {FBCLIP_CHECKPOINT}")
print(f"CLIP backbone:      {FBCLIP_CLIP_WEIGHT}")
print(f"Zero corruption:    {ZERO_CORRUPTION_CONDITION if INCLUDE_CLEAN_BASELINE else 'disabled'}")
print(f"Corruptions:        {CORRUPTION_TYPES} @ {SEVERITY_LEVELS}")
print(f"Categorized:        {USE_CATEGORIZED_CORRUPTIONS}")
print(f"Plan:               {CORRUPTION_PLAN}")
print(f"Device/batch:       {DEVICE} / {BATCH_SIZE}")
print(f"Outputs:            {OUTPUT_ROOT}")

run_evaluation(
    mvtec_root=MVTEC_PATH if IS_MVTEC else None,
    visa_root=None if IS_MVTEC else VISA_PATH,
    output_root=OUTPUT_ROOT,
    models=[MODEL_NAME],
    model_kwargs=model_kwargs,
    device=DEVICE,
    dataset=DATASET_NAME,
    corruption_types=CORRUPTION_TYPES,
    severity_levels=SEVERITY_LEVELS,
    include_clean=INCLUDE_CLEAN_BASELINE,
    batch_size=BATCH_SIZE,
    corruption_cache_root=CORRUPTION_CACHE_ROOT,
    corruption_cache_format=CORRUPTION_CACHE_FORMAT,
    categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
    categorized_corruption_plans={
        DATASET_NAME: str(CORRUPTION_PLAN)
    },
    corruption_seed=(
        CATEGORIZED_CORRUPTION_SEED
        if USE_CATEGORIZED_CORRUPTIONS
        else None
    ),
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"FB-CLIP evaluation complete. Outputs: {OUTPUT_ROOT}")